In [ ]:
import tidy3d as td
import numpy as np
import tidy3d.web as web
import matplotlib.pyplot as plt
from tidy3d import material_library
from tidy3d.plugins.mode import ModeSolver
import pandas as pd


# MMI 2x2 SiN 300 nm con sobrecapa BCB

Copia del caso 8515 con `mmi_length = 114.71 um`, `mmi_width = 10 um` y core SiN de 300 nm. El stack confirmado es: SiO2 desde el fondo de la ventana hasta `z=0`, core SiN de `z=0` a `0.3 um`, BCB de `z=0` a `0.4 um` y aire por encima. La capa BCB cubre todo XY, por lo que deja una cubierta de 100 nm sobre el core.

BCB usa `n=1.5779` constante y `k=0` (sin perdidas), convertido a `epsilon_r = n**2`. Se conserva el modelo Si3N4 del core. Para la ventana vertical uso 1.55 um de SiO2 debajo y 1.55 um de aire encima del BCB.

**Prioridades en el solapamiento:** SiO2 = 0, BCB = 1 y SiN = 2. Asi el core SiN prevalece sobre el BCB en el volumen compartido; el BCB queda por encima del core en los 100 nm restantes. La prioridad mayor determina el material en la region solapada, como describe la [documentacion oficial de Tidy3D](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.Structure.html).


In [ ]:
wl_c = 1.55
mmi_length = 113.71
mmi_width = 10.0
wg_gap = 1.0 
wg_width = 1.0
wg_thick = 0.3
wg_offset = 1.5
wg_length = 1
wg_mmi_dist = 8
mmi_input_width = 4.0
mmi_input_length = 50.0
port_y = wg_gap / 2 + mmi_input_width / 2
bcb_thickness = 0.4
substrate_thickness = wl_c
air_thickness = wl_c
z_min = -substrate_thickness
z_max = bcb_thickness + air_thickness
center_z = (z_min + z_max) / 2
core_center_z = wg_thick / 2
size_z = z_max - z_min
size_y = wg_offset+mmi_width+2*wl_c
size_x = mmi_length+2*mmi_input_length+2*wg_mmi_dist+2*wg_length
SiO2 = td.Medium(
    name = 'SiO2', 
    permittivity = 2.085136, 
)

# Stoichiometric silicon nitride core, dispersive model from the official library.
sin_n = 1.9729
SiN = td.Medium(name="SiN", permittivity=sin_n**2)

# Constant BCB refractive index supplied for this model; k=0 is assumed.
bcb_n = 1.5779
BCB = td.Medium(
    name="BCB",
    permittivity=bcb_n**2,
    viz_spec=td.VisualizationSpec(facecolor="#E6A15A", edgecolor="#80511F", alpha=0.0),
)
Air = td.Medium(name="Air", permittivity=1.0)


## Definir la simulacion

Esta celda construye la geometria, fuentes, monitores y el objeto `sim`. No lanza el solver.


In [ ]:
modesource_0 = td.ModeSource(
    name = 'modesource_0', 
    center = [-size_x / 2 + wg_length-0.225, port_y, core_center_z], 
    size = [0, 4 * wg_width, 5 * wg_thick], 
    source_time = td.GaussianPulse(freq0 = 193615962458333.3, fwidth = 12491352416666.656, ), 
    num_freqs = 5, 
    direction = '+', 
    mode_spec = td.ModeSpec(num_modes = 4, sort_spec = {'filter_reference' : 0, 'filter_order':'over', 'sort_key':'n_eff', 'track_freq':'central', 'keep_modes':'all'}, group_index_step = True, ), 
)

monitor_S31 = td.ModeMonitor(
    name = 'monitor_S31', 
    center = [-(-size_x / 2 + wg_length-0.225), port_y, core_center_z], 
    size = [0, 4 * wg_width, 5 * wg_thick], 
    freqs = td.C_0 / np.linspace(1.5000000000000002, 1.6, 51), 
    mode_spec = td.ModeSpec(num_modes = 1, sort_spec = {'filter_reference' : 0, 'filter_order':'over', 'sort_key':'n_eff', 'track_freq':'central', 'keep_modes':'all'}, group_index_step = True, )
)

monitor_S41 = td.ModeMonitor(
    name = 'monitor_S41', 
    center = [-(-size_x / 2 + wg_length-0.225), -(port_y), core_center_z], 
    size = [0, 4 * wg_width, 5 * wg_thick], 
    freqs = td.C_0 / np.linspace(1.5000000000000002, 1.6, 51), 
    mode_spec = td.ModeSpec(num_modes = 1, sort_spec = {'filter_reference' : 0, 'filter_order':'over', 'sort_key':'n_eff', 'track_freq':'central', 'keep_modes':'all'}, group_index_step = True, )
)

field_xy = td.FieldMonitor(
    name = 'field_xy', 
    center = [0, 0, core_center_z], 
    size = [td.inf, td.inf, 0], 
    freqs = td.C_0 / 1.5500000000000003, 
)

field_3d = td.FieldMonitor(
    name = 'field_3d', 
    center = [0, 0, core_center_z], 
    size = [td.inf, td.inf, wg_thick], 
    freqs = td.C_0 / 1.5500000000000003, 
)

substrate = td.Structure(
    geometry=td.Box(
        center=[0, 0, z_min + substrate_thickness / 2],
        size=[td.inf, td.inf, substrate_thickness],
    ),
    name="SiO2_substrate",
    medium=SiO2,
    priority=0,
)

bcb_layer = td.Structure(
    geometry=td.Box(
        center=[0, 0, bcb_thickness / 2],
        size=[td.inf, td.inf, bcb_thickness],
    ),
    name="BCB_overcladding",
    medium=BCB,
    priority=1,
)

mmi_1 = td.Structure(
    geometry = td.PolySlab(slab_bounds = [0, wg_thick], vertices = [[-mmi_length / 2-mmi_input_length, wg_gap / 2 + mmi_input_width / 2 + wg_width / 2], [-mmi_length / 2, wg_gap / 2 + mmi_input_width], [-mmi_length / 2, wg_gap / 2], [-mmi_length / 2-mmi_input_length, wg_gap / 2 + mmi_input_width / 2-wg_width / 2]]), 
    name = 'mmi_1', 
    medium = SiN,
    priority = 2
)

mmi_2 = td.Structure(
    geometry = td.PolySlab(slab_bounds = [0, wg_thick], vertices = [[-mmi_length / 2-mmi_input_length, -(wg_gap / 2 + mmi_input_width / 2 + wg_width / 2)], [-mmi_length / 2, -(wg_gap / 2 + mmi_input_width)], [-mmi_length / 2, -(wg_gap / 2)], [-mmi_length / 2-mmi_input_length, -(wg_gap / 2 + mmi_input_width / 2-wg_width / 2)]]), 
    name = 'mmi_2', 
    medium = SiN,
    priority = 2
)

mmi_3 = td.Structure(
    geometry = td.PolySlab(slab_bounds = [0, wg_thick], vertices = [[-(-mmi_length / 2-mmi_input_length), wg_gap / 2 + mmi_input_width / 2 + wg_width / 2], [-(-mmi_length / 2), wg_gap / 2 + mmi_input_width], [-(-mmi_length / 2), wg_gap / 2], [-(-mmi_length / 2-mmi_input_length), wg_gap / 2 + mmi_input_width / 2-wg_width / 2]]), 
    name = 'mmi_3', 
    medium = SiN,
    priority = 2
)

mmi_4 = td.Structure(
    geometry = td.PolySlab(slab_bounds = [0, wg_thick], vertices = [[-(-mmi_length / 2-mmi_input_length), -(wg_gap / 2 + mmi_input_width / 2 + wg_width / 2)], [-(-mmi_length / 2), -(wg_gap / 2 + mmi_input_width)], [-(-mmi_length / 2), -(wg_gap / 2)], [-(-mmi_length / 2-mmi_input_length), -(wg_gap / 2 + mmi_input_width / 2-wg_width / 2)]]), 
    name = 'mmi_4', 
    medium = SiN,
    priority = 2
)

mmi_slab = td.Structure(
    geometry=td.Box(
        center=[0, 0, core_center_z],
        size=[mmi_length, mmi_width, wg_thick],
    ),
    name="mmi_slab",
    medium=SiN,
    priority=2,
)

# Straight access waveguides extend each taper to the simulation ports.
# wg_mmi_dist sets the taper-to-boundary distance; wg_length adds a short
# section beyond the simulation boundary for each access guide.
port_x_left = -mmi_length / 2 - mmi_input_length
port_x_right = mmi_length / 2 + mmi_input_length
access_length = wg_mmi_dist + 2 * wg_length
access_x = port_x_right + access_length / 2

wg_1 = td.Structure(
    geometry=td.Box(center=[access_x, port_y, core_center_z], size=[access_length, wg_width, wg_thick]),
    name="wg_1", medium=SiN, priority=2
)
wg_2 = td.Structure(
    geometry=td.Box(center=[-access_x, port_y, core_center_z], size=[access_length, wg_width, wg_thick]),
    name="wg_2", medium=SiN, priority=2
)
wg_3 = td.Structure(
    geometry=td.Box(center=[access_x, -port_y, core_center_z], size=[access_length, wg_width, wg_thick]),
    name="wg_3", medium=SiN, priority=2
)
wg_4 = td.Structure(
    geometry=td.Box(center=[-access_x, -port_y, core_center_z], size=[access_length, wg_width, wg_thick]),
    name="wg_4", medium=SiN, priority=2
)

sim = td.Simulation(
    center = [0, 0, center_z],
    size = [size_x, size_y, size_z], 
    grid_spec = td.GridSpec.auto(wavelength = 1.5500000000000003, min_steps_per_wvl = 15), 
    run_time = {'quality_factor' : 1, 'source_factor' : 3},
    medium = Air,
    sources = [modesource_0],
    monitors = [monitor_S31, monitor_S41, field_xy, field_3d],
    structures = [substrate, bcb_layer, mmi_1, mmi_2, mmi_3, mmi_4, mmi_slab, wg_1, wg_2, wg_3, wg_4],
)
# La simulacion queda definida en `sim`; se ejecutara en una celda posterior.


## Comprobar el indice efectivo de la guia de entrada

Se resuelve localmente una seccion recta de la guia de entrada, entre el puerto y el taper. El plano modal incluye el stack vertical del modelo: sustrato SiO2, core SiN, BCB y aire. Se muestran los cuatro primeros modos y sus `n_eff`; el modo 0 es el primero que devuelve el solver.

`ModeSolver.solve()` se ejecuta localmente, sin lanzar otro trabajo FDTD en la nube ni consumir FlexCredits. La documentacion advierte que el solver local puede no incluir subpixel-smoothing.

[API oficial de ModeSolver](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.plugins.mode.ModeSolver.html) ? [Guia oficial de configuracion](https://docs.flexcompute.com/projects/tidy3d/en/v2.12.0/faq/docs/faq/how-do-i-set-the-mode-solver.html).


In [ ]:
# Centro de la seccion recta de acceso de la guia de entrada.
access_length = wg_mmi_dist + 2 * wg_length
mode_x = -mmi_length / 2 - mmi_input_length - access_length / 2
mode_plane = td.Box(
    center=[mode_x, port_y, center_z],
    size=[0, 4 * wg_width, size_z],
)

mode_spec = td.ModeSpec(num_modes=4)
mode_solver = ModeSolver(
    simulation=sim,
    plane=mode_plane,
    mode_spec=mode_spec,
    freqs=[td.C_0 / wl_c],
)
mode_data = mode_solver.solve()

n_eff_modes = np.real(np.asarray(mode_data.n_eff.values).reshape(-1))
mode_table = pd.DataFrame({
    "mode_index": np.arange(n_eff_modes.size),
    "n_eff": n_eff_modes,
})
display(mode_table)
print(f"n_eff del modo 0 a {wl_c:.3f} um: {n_eff_modes[0]:.6f}")


## Estimar el coste

La estimacion devuelve el coste maximo en FlexCredits y no ejecuta el solver. Si modificas el modelo, vuelve a ejecutar primero la celda de definicion y despues esta celda.


API oficial: [web.Job y estimate_cost](https://docs.flexcompute.com/projects/tidy3d/en/latest/api/_autosummary/tidy3d.web.Job.html).


In [ ]:
task_name = "2x2_SiN300_BCB_8515_FDTD"

job = web.Job(
    simulation=sim,
    task_name=task_name,
)

estimated_cost = job.estimate_cost()
print(f"Coste maximo estimado: {estimated_cost:.3f} FlexCredits")


## Ejecutar la simulacion

Ejecuta esta celda solo cuando quieras iniciar el calculo. Usa el mismo `job` creado en la celda de estimacion.


In [ ]:
sim_data = job.run(path="./data/" + task_name + ".hdf5")


## Resultados

Primero se muestra el campo electrico total en el monitor XY. Despues se calculan las razones de potencia y el Excess Loss en dB.


API oficial para visualizar datos: [plot_field](https://docs.flexcompute.com/projects/tidy3d/en/latest/faq/docs/faq/how-do-i-plot-the-simulation-structure-on-top-of-my-field-plot.html).


In [ ]:
ax = sim_data.plot_field(
    field_monitor_name="field_xy",
    field_name="E",
    val="abs",
    z=core_center_z,
)
ax.set_title("Magnitud del campo electrico |E| en el plano XY")
plt.show()


In [ ]:
# Amplitudes complejas del modo 0 que propaga hacia +x
S31 = sim_data["monitor_S31"].amps.sel(direction="+", mode_index=0)
S41 = sim_data["monitor_S41"].amps.sel(direction="+", mode_index=0)

# Potencias modales normalizadas
P31 = np.abs(S31)**2
P41 = np.abs(S41)**2


total_power = P31 + P41
excess_loss_db = -10 * np.log10(total_power)

# Longitud de onda en nm: td.C_0 está expresada en µm/s
f = P31.f.values
wavelength_nm = 1000 * td.C_0 / P31.f.values

fig, ax = plt.subplots(2, 1, sharex=True, tight_layout=True)
ax[0].plot(td.C_0/f, P31/(P31+P41), label=r"S31")
ax[0].plot(td.C_0/f, P41/(P31+P41), label=r"S41")
ax[0].legend()
ax[1].plot(td.C_0/f, excess_loss_db, 'k')

ax[0].set(ylabel=r"$|S|^2$")
ax[1].set(xlabel=r"Wavelength (um)", ylabel="Excess loss (dB)", xlim=(1.5,1.6))

ax[0].set_title("Power split ratio")
ax[1].set_title("Excess loss")

plt.show()

## Resumen del MMI

Convencion para la excitacion del puerto 1 superior: `monitor_S31` (salida superior) se toma como salida directa y `monitor_S41` (salida inferior) como salida cruzada. Si tu numeracion fisica asigna los puertos de otra forma, intercambia esas dos asignaciones.

Se calcula `K = P_cruzada / (P_cruzada + P_directa)`, `1-K`, la perdida excedente `-10 log10(P_cruzada + P_directa)` en dB y la fase relativa `fase_cruzada - fase_directa` en grados. La fuente `ModeSource` de Tidy3D esta normalizada a 1 W; consulta la [documentacion oficial de normalizacion](https://docs.flexcompute.com/projects/tidy3d/en/latest/faq/docs/faq/how-are-results-normalized.html).

El resumen impreso corresponde al punto de frecuencia mas cercano a `wl_c`; los arrays calculados conservan los valores en toda la banda.


La ejecucion anterior de silicio tuvo un decaimiento final superior al umbral de parada. Comprueba el log de esta estructura SiN-BCB; si vuelve a aparecer la advertencia, trata los resultados como provisionales y ajusta `run_time`.


La estimacion modal se realiza con el modelo Si3N4 seleccionado. Verifica los modos en la ejecucion antes de interpretar los resultados.


In [ ]:
# Mapping for input port 1: upper output is direct (S31), lower output is cross (S41).
p_direct = np.asarray(P31.values, dtype=float)
p_cross = np.asarray(P41.values, dtype=float)
p_total = p_direct + p_cross

K = np.divide(p_cross, p_total, out=np.full_like(p_total, np.nan), where=p_total > 0)
one_minus_K = 1.0 - K
excess_loss_db = np.full_like(p_total, np.nan)
positive_total = p_total > 0
excess_loss_db[positive_total] = -10 * np.log10(p_total[positive_total])

# Wrapped phase difference in [-180, 180] degrees: phase(S41) - phase(S31).
phase_relation_deg = np.rad2deg(np.angle(S41.values * np.conj(S31.values)))
wavelength_um = td.C_0 / f
nominal_index = int(np.argmin(np.abs(wavelength_um - wl_c)))

print(f"MMI response near wl_c = {wavelength_um[nominal_index]:.4f} um")
print(f"K (cross-output power ratio):       {K[nominal_index]:.6f} ({100*K[nominal_index]:.3f}%)")
print(f"1-K (direct-output power ratio):    {one_minus_K[nominal_index]:.6f} ({100*one_minus_K[nominal_index]:.3f}%)")
print(f"Excess loss:                        {excess_loss_db[nominal_index]:.4f} dB")
print(f"Output phase relation (cross-direct): {phase_relation_deg[nominal_index]:.2f} deg")
